# Resultados de Classificação — Pós-Processamento e Visualização

Este notebook foi criado pela [biodiversica](https://biodiversica.xyz) e carrega resultados de detecção de qualquer ferramenta de classificação de áudio, mescla-os, gera gráficos e extrai clipes de áudio para as detecções de interesse.

---

### O que este notebook faz (em ordem):
1. **Conecta ao seu Google Drive** para ler resultados e salvar saídas
2. **Carrega e mescla** todos os arquivos de resultado de uma pasta escolhida
3. **Filtra** os resultados por limiar de confiança e espécie
4. **Gera gráficos** — mapas de calor, gráficos de dispersão, gráficos de riqueza de espécies
5. **Extrai segmentos de áudio** — clipes curtos para cada detecção, provenientes do Arbimon ou do seu Drive
6. **Revisa os segmentos extraídos** — ouça cada clipe, visualize seu espectrograma e marque como verdadeiro ou falso

### Antes de começar:
- Seus arquivos de resultado devem ser CSV ou TXT com uma detecção por linha
- Você precisa de uma **conta Google** com Google Drive

### Como executar:
Execute cada célula **uma de cada vez**, de cima para baixo. Clique no botão ▶ ou pressione `Shift + Enter`.

> **Novo em notebooks?** Uma célula com `[ ]` ainda não foi executada. Após executar, aparece um número como `[1]`. Texto vermelho significa erro — leia a mensagem, ela geralmente indica exatamente o que corrigir.

---
### Índice
- [Passo 1 — Conectar ao Google Drive e Instalar Software](#step-1)
- [Passo 2 — Carregar e Mesclar Resultados](#step-2)
- [Passo 3 — Filtrar Resultados](#step-3)
- [Passo 4 — Gerar Gráficos](#step-4)
- [Passo 5 — Extrair Segmentos de Áudio](#step-5)
- [Passo 6 — Revisar Segmentos Extraídos](#step-6)

---
## Passo 1 — Conectar ao Google Drive e Instalar Software <a name="step-1"></a>

Execute as duas células abaixo. A primeira pedirá que você **permita o acesso ao seu Google Drive** — clique no link e siga os passos.

In [1]:
#@title 📂 Conectar ao Google Drive { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive conectado com sucesso.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive connected successfully.


In [2]:
#@title 📦 Instalar pacotes { display-mode: "form" }

!pip install pandas matplotlib seaborn librosa soundfile -q

!wget -q https://github.com/rfcx/rfcx-sdk-python/releases/download/0.3.1/rfcx-0.3.1-py3-none-any.whl \
    -O /tmp/rfcx-0.3.1-py3-none-any.whl
!pip install /tmp/rfcx-0.3.1-py3-none-any.whl -q

print('All packages installed successfully.')

All packages installed successfully.


---
## Passo 2 — Carregar e Mesclar Resultados <a name="step-2"></a>

Aponte o notebook para a pasta no seu Google Drive onde os arquivos de resultado estão armazenados.
A pasta pode ter uma subpasta por ponto de gravação — todos os arquivos `.txt` e `.csv` dentro são encontrados e mesclados automaticamente.

In [ ]:
#@title 📁 Pasta de resultados { display-mode: "form" }
#@markdown Caminho completo para a pasta no seu Google Drive que contém os arquivos de resultado.
#@markdown Subpastas (uma por ponto de gravação) são pesquisadas automaticamente.
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/arbimon_birdnet_results"  #@param {type:"string"}

import os, glob as _g
if not os.path.isdir(DRIVE_RESULTS_DIR):
    print(f"WARNING: Folder not found: {DRIVE_RESULTS_DIR}")
    print("Check the path above and make sure Google Drive is connected (Step 1).")
else:
    n = len(_g.glob(f'{DRIVE_RESULTS_DIR}/**/*.txt', recursive=True) +
            _g.glob(f'{DRIVE_RESULTS_DIR}/**/*.csv', recursive=True))
    print(f"Folder found   : {DRIVE_RESULTS_DIR}")
    print(f"Result files   : {n} file(s) detected")

In [ ]:
#@title ⚙️ Formato do arquivo e configurações de colunas { display-mode: "form" }
#@markdown Configure como seus arquivos de resultado estão estruturados.
#@markdown Os padrões correspondem à saída do **Analisador de Áudio do Arbimon**.
#@markdown
#@markdown ---
#@markdown **Formato do arquivo** — extensão a buscar na pasta de resultados.
FILE_FORMAT = "csv"  #@param ["csv", "txt"]

#@markdown **Delimitador** — caractere que separa colunas em cada linha.
DELIMITER = "comma (,)"  #@param ["comma (,)", "tab", "semicolon (;)"]

#@markdown ---
#@markdown ### Nomes das colunas
#@markdown Insira o cabeçalho exato da coluna como aparece nos seus arquivos de resultado.
#@markdown Deixe em branco para colunas opcionais que seus arquivos não contêm.

#@markdown **Espécie / rótulo**
COL_LABEL = "scientific_name"  #@param {type:"string"}

#@markdown **Ponto de gravação**
COL_SITE = "site"  #@param {type:"string"}

#@markdown **Data** *(AAAA-MM-DD)*
COL_DATE = "date"  #@param {type:"string"}

#@markdown **Hora** *(HH:MM:SS)*
COL_TIME = "time"  #@param {type:"string"}

#@markdown **Pontuação de confiança**
COL_CONFIDENCE = "confidence"  #@param {type:"string"}

#@markdown **Tempo de início da detecção** *(segundos dentro da gravação)*
COL_START_TIME = "start_time"  #@param {type:"string"}

#@markdown **Tempo de fim da detecção** *(segundos dentro da gravação)*
COL_END_TIME = "end_time"  #@param {type:"string"}

#@markdown ---
#@markdown *As duas colunas abaixo só são necessárias ao extrair áudio do Arbimon.*

#@markdown **Stream / ID de gravação** *(somente Arbimon)*
COL_STREAM_ID = "stream_id"  #@param {type:"string"}

#@markdown **Fuso horário UTC** *(somente Arbimon — ex.: UTC-3)*
COL_UTC_OFFSET = "utc_offset"  #@param {type:"string"}

print('Configurações de formato salvas. Execute a próxima célula para carregar os arquivos.')

In [ ]:
#@title 📊 Carregar e mesclar todos os arquivos de resultado { display-mode: "form" }
#@markdown Lê todos os arquivos de resultado na sua pasta e os combina em uma única tabela.

import os, glob, warnings
import pandas as pd
warnings.filterwarnings('ignore')

_delim_map = {"comma (,)": ",", "tab": "\t", "semicolon (;)": ";"}
_sep = _delim_map.get(DELIMITER, ",")

result_files = glob.glob(f'{DRIVE_RESULTS_DIR}/**/*.{FILE_FORMAT}', recursive=True)

if not result_files:
    raise FileNotFoundError(
        f'No .{FILE_FORMAT} files found in: {DRIVE_RESULTS_DIR}\n'
        'Check the folder path, the file format setting above, and make sure the result files are there.'
    )

dfs = []
for f in result_files:
    try:
        tmp = pd.read_csv(f, sep=_sep)
        if not tmp.empty:
            dfs.append(tmp)
    except Exception as e:
        print(f'  WARNING: could not read {os.path.basename(f)}: {e}')

if not dfs:
    raise ValueError('All result files were empty or unreadable.')

df_raw = pd.concat(dfs, ignore_index=True)

# Build rename map: user column name → internal standard name
_required_cols = {
    COL_LABEL:      'Common name',
    COL_SITE:       'Point name',
    COL_DATE:       'Date',
    COL_TIME:       'Time',
    COL_CONFIDENCE: 'Confidence',
    COL_START_TIME: 'start_time',
    COL_END_TIME:   'end_time',
}
_optional_cols = {}
if COL_STREAM_ID.strip():  _optional_cols[COL_STREAM_ID]  = 'stream_id'
if COL_UTC_OFFSET.strip(): _optional_cols[COL_UTC_OFFSET] = 'utc_offset'

_rename = {}
for src, dst in _required_cols.items():
    if not src.strip():
        print(f'  WARNING: no column name set for "{dst}" — check the format settings.')
    elif src not in df_raw.columns:
        print(f'  WARNING: column "{src}" not found in files (expected for "{dst}").')
    else:
        _rename[src] = dst

for src, dst in _optional_cols.items():
    if src in df_raw.columns:
        _rename[src] = dst

df_raw = df_raw.rename(columns=_rename)

df_raw['Date']       = pd.to_datetime(df_raw['Date'], errors='coerce').dt.date
df_raw['Time']       = pd.to_datetime(df_raw['Time'], format='%H:%M:%S', errors='coerce').dt.time
df_raw['Confidence'] = pd.to_numeric(df_raw['Confidence'], errors='coerce')
if 'start_time' in df_raw.columns:
    df_raw['start_time'] = pd.to_numeric(df_raw['start_time'], errors='coerce')
if 'end_time' in df_raw.columns:
    df_raw['end_time'] = pd.to_numeric(df_raw['end_time'], errors='coerce')

print(f'Arquivos carregados    : {len(result_files)}')
print(f'Total de detecções     : {len(df_raw)}')
print(f'Pontos de gravação     : {sorted(df_raw["Point name"].dropna().unique().tolist())}')
print(f'Rótulos únicos         : {df_raw["Common name"].nunique()}')
print(f'Intervalo de datas     : {df_raw["Date"].min()} → {df_raw["Date"].max()}')
print(f'Intervalo de confiança : {df_raw["Confidence"].min():.3f} → {df_raw["Confidence"].max():.3f}')

---
## Passo 3 — Filtrar Resultados <a name="step-3"></a>

Defina o limiar mínimo de confiança e escolha quais espécies incluir ou excluir.
Execute **Aplicar filtros** depois — ele mostrará um resumo do que resta.

In [ ]:
#@title 🔬 Configurações de filtro { display-mode: "form" }

#@markdown **Minimum confidence** — detections below this score are discarded.
#@markdown Menor = mais detecções (pode incluir falsos positivos). Maior = menos, porém mais confiáveis.
MIN_CONFIDENCE = 0.5  #@param {type:"slider", min:0.0, max:1.0, step:0.05}

#@markdown ---
#@markdown **Incluir todas as espécies?** — se marcado, todos os rótulos nos resultados são usados.
#@markdown Desmarque para listar espécies específicas no campo abaixo.
ALL_SPECIES = True  #@param {type:"boolean"}

#@markdown **Espécies a incluir** — usado apenas quando "Incluir todas as espécies" está desmarcado.
#@markdown Separe os nomes com vírgulas. Exemplo: `Robin, Blackbird, Chiffchaff`
SPECIES_INCLUDE = ""  #@param {type:"string"}

#@markdown ---
#@markdown **Espécies a excluir** — sempre aplicado, mesmo quando "Incluir todas as espécies" está marcado.
#@markdown Útil para remover rótulos de ruído de fundo ou indesejados.
#@markdown Separe os nomes com vírgulas. Exemplo: `Background noise, Unknown`
SPECIES_EXCLUDE = ""  #@param {type:"string"}

print('Configurações salvas. Execute a próxima célula para aplicá-las.')

In [ ]:
#@title ✅ Aplicar filtros { display-mode: "form" }
#@markdown Execute esta célula após alterar qualquer configuração de filtro acima.
#@markdown Ela mostra um resumo das detecções restantes por espécie.

def _split_list(s):
    return [x.strip() for x in s.split(',') if x.strip()]

df = df_raw.copy()
df = df[df['Confidence'] >= MIN_CONFIDENCE]

exclude_list = _split_list(SPECIES_EXCLUDE)
if exclude_list:
    df = df[~df['Common name'].isin(exclude_list)]

if ALL_SPECIES:
    target_species = sorted(df['Common name'].dropna().unique().tolist())
else:
    target_species = _split_list(SPECIES_INCLUDE)
    if not target_species:
        print('AVISO: "Incluir todas as espécies" está desativado mas nenhum nome foi listado — incluindo todas.')
        target_species = sorted(df['Common name'].dropna().unique().tolist())
    else:
        df = df[df['Common name'].isin(target_species)]

df['Common name'] = pd.Categorical(df['Common name'], categories=target_species, ordered=True)

print(f'Limiar de confiança  : ≥ {MIN_CONFIDENCE}')
print(f'Detecções restantes  : {len(df)}')
print(f'Espécies incluídas   : {len(target_species)}')
for sp in target_species:
    n = int((df['Common name'] == sp).sum())
    print(f'  {sp}: {n} detecção(ões)')
if exclude_list:
    print(f'Espécies excluídas   : {exclude_list}')

---
## Passo 4 — Gerar Gráficos <a name="step-4"></a>

Escolha quais gráficos gerar, defina uma pasta de saída e execute a última célula.
Todos os gráficos são salvos como arquivos PNG de alta resolução no seu Google Drive.

| Gráfico | O que mostra |
|---------|--------------|
| **Mapa de calor — espécie × ponto** | Contagem de detecções por espécie em cada ponto de gravação (todas as datas combinadas) |
| **Mapa de calor — espécie × ponto por dia** | Mesmo mapa de calor dividido em um painel por dia de gravação |
| **Dispersão — confiança vs. hora (todos os pontos)** | Todos os pontos em uma figura, um subgráfico por ponto |
| **Dispersão — confiança vs. hora (por ponto)** | Uma figura separada para cada ponto de gravação |
| **Mapa de calor — espécie × hora do dia** | Qual hora do dia cada espécie foi mais ativa |
| **Gráfico de barras — riqueza de espécies** | Contagem total e exclusiva de espécies por ponto |

> Execute os Passos 2 e 3 primeiro para que os dados estejam carregados e filtrados antes de gerar os gráficos.

In [ ]:
#@title 📊 Escolher gráficos { display-mode: "form" }
#@markdown Marque cada gráfico que deseja gerar.

PLOT_HEATMAP_ALL = True  #@param {type:"boolean"}
#@markdown **Mapa de calor — contagem de detecções por espécie e ponto (todas as datas)**

PLOT_HEATMAP_PER_DAY = False  #@param {type:"boolean"}
#@markdown **Mapa de calor — igual ao anterior, um painel por dia de gravação**

PLOT_SCATTER_CONFIDENCE = False  #@param {type:"boolean"}
#@markdown **Dispersão — confiança vs. hora do dia (todos os pontos em uma figura)**

PLOT_MULTIPLE_SCATTER = True  #@param {type:"boolean"}
#@markdown **Dispersão — confiança vs. hora do dia (uma figura por ponto)**

PLOT_HEATMAP_SPECIES_HOUR = True  #@param {type:"boolean"}
#@markdown **Mapa de calor — detecções por espécie por hora do dia**

PLOT_RICHNESS = True  #@param {type:"boolean"}
#@markdown **Gráfico de barras — riqueza de espécies por ponto (únicas vs total)**

#@markdown ---
#@markdown **Limite do eixo Y para o gráfico de riqueza de espécies**
MAX_SPECIES_AXIS = 20  #@param {type:"integer"}

print('Seleção de gráficos salva. Defina a pasta de saída e execute os gráficos na próxima célula.')

In [ ]:
#@title 🎨 Pasta de saída e executar gráficos { display-mode: "form" }
#@markdown Todos os gráficos selecionados são salvos como arquivos PNG nesta pasta do seu Google Drive.
DRIVE_PLOTS_DIR = "/content/drive/MyDrive/arbimon_plots"  #@param {type:"string"}

import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import PowerNorm

os.makedirs(DRIVE_PLOTS_DIR, exist_ok=True)

def _time_float(t):
    return t.hour + t.minute / 60.0 + t.second / 3600.0 if pd.notnull(t) else None

generated = []

# ── Heatmap: species × site ───────────────────────────────────────────────────
if PLOT_HEATMAP_ALL:
    print('Gerando: mapa de calor — espécie × ponto...')
    cnt = df.groupby(['Common name', 'Point name']).size().reset_index(name='Count')
    hm  = cnt.pivot(index='Common name', columns='Point name', values='Count').fillna(0)
    hm  = hm.reindex(sorted(hm.index))[sorted(hm.columns)]
    plt.figure(figsize=(12, max(4, len(hm) * 0.35)))
    sns.heatmap(hm, annot=True, fmt='.0f', cmap='YlGnBu', linewidths=0.5, cbar=False,
                norm=PowerNorm(gamma=0.25, vmin=0, vmax=max(hm.values.max(), 1)))
    plt.title(f'Detecções por espécie e ponto  (Confiança ≥ {MIN_CONFIDENCE})', fontsize=13, fontweight='bold')
    plt.xlabel('Ponto'); plt.ylabel('Espécie')
    plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
    plt.tight_layout()
    out = f'{DRIVE_PLOTS_DIR}/heatmap_species_vs_sites.png'
    plt.savefig(out, dpi=300); plt.show(); plt.close(); generated.append(out)
    print('  Salvo: heatmap_species_vs_sites.png')

# ── Heatmap: species × site per day ──────────────────────────────────────────
if PLOT_HEATMAP_PER_DAY:
    print('Gerando: mapa de calor — por dia...')
    unique_dates = sorted(df['Date'].dropna().unique())
    if not unique_dates:
        print('  Nenhuma informação de data encontrada — ignorando.')
    else:
        cnt    = df.groupby(['Date', 'Common name', 'Point name']).size().reset_index(name='Count')
        sp_all = sorted(df['Common name'].unique())
        pts    = sorted(df['Point name'].unique())
        n      = len(unique_dates)
        fig, axes = plt.subplots(1, n, figsize=(6 * n, max(4, 0.35 * len(sp_all))))
        if n == 1: axes = [axes]
        for i, (ax, d) in enumerate(zip(axes, unique_dates)):
            hm = (cnt[cnt['Date'] == d]
                  .pivot(index='Common name', columns='Point name', values='Count')
                  .reindex(index=sp_all, columns=pts).fillna(0))
            sns.heatmap(hm, ax=ax, annot=True, fmt='.0f', cmap='YlGnBu',
                        linewidths=0.5, linecolor='black', cbar=False, vmin=0,
                        norm=PowerNorm(gamma=0.25, vmin=0, vmax=max(hm.values.max(), 1)))
            ax.set_title(str(d), fontsize=11, fontweight='bold')
            ax.set_xlabel('Ponto')
            ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
            if i == 0:
                ax.set_ylabel('Espécie'); ax.set_yticklabels(sp_all, rotation=0)
            else:
                ax.set_ylabel(''); ax.set_yticklabels([])
        plt.tight_layout(); plt.subplots_adjust(wspace=0.025)
        out = f'{DRIVE_PLOTS_DIR}/heatmap_species_vs_sites_per_day.png'
        plt.savefig(out, dpi=300); plt.show(); plt.close(); generated.append(out)
        print('  Salvo: heatmap_species_vs_sites_per_day.png')

# ── Scatter setup (shared between both scatter plots) ─────────────────────────
if PLOT_SCATTER_CONFIDENCE or PLOT_MULTIPLE_SCATTER:
    _sc = df.copy()
    _sc['Date']       = pd.to_datetime(_sc['Date'], errors='coerce')
    _sc['time_float'] = _sc['Time'].apply(_time_float)
    palette   = sns.color_palette('tab10', max(len(target_species), 1))
    color_map = {sp: palette[i % len(palette)] for i, sp in enumerate(target_species)}
    leg_handles = [
        plt.Line2D([], [], color=color_map[sp], marker='o', linestyle='', markersize=6, label=sp)
        for sp in target_species
    ]

# ── Scatter: all sites in one figure ─────────────────────────────────────────
if PLOT_SCATTER_CONFIDENCE:
    print('Gerando: dispersão — todos os pontos em uma figura...')
    pts   = sorted(_sc['Point name'].unique())
    ncols = 2
    nrows = int(np.ceil(len(pts) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
    for ax in axes[len(pts):]: ax.set_visible(False)
    for ax, p in zip(axes, pts):
        sub = _sc[_sc['Point name'] == p]
        for sp in sub['Common name'].unique():
            s = sub[sub['Common name'] == sp]
            ax.scatter(s['time_float'], s['Confidence'],
                       color=color_map.get(str(sp), 'gray'), marker='o', s=40, alpha=0.8)
        ax.set_xticks(np.arange(0, 25, 1))
        ax.grid(True, linestyle='--', color='gray', alpha=0.4); ax.set_axisbelow(True)
        ax.set_title(str(p), fontsize=10, fontweight='bold')
        ax.set_xlabel('Horário'); ax.set_ylabel('Confiança')
        ax.set_xlim(0, 24); ax.set_ylim(MIN_CONFIDENCE, 1.0)
    fig.legend(leg_handles, target_species, loc='upper center', ncol=4,
               bbox_to_anchor=(0.5, 1.0), fontsize=9, frameon=False)
    plt.tight_layout(rect=[0, 0, 1, 0.9])
    out = f'{DRIVE_PLOTS_DIR}/scatter_confidence_all_sites.png'
    plt.savefig(out, dpi=300); plt.show(); plt.close(); generated.append(out)
    print('  Salvo: scatter_confidence_all_sites.png')

# ── Scatter: one figure per site ─────────────────────────────────────────────
if PLOT_MULTIPLE_SCATTER:
    print('Gerando: dispersão — uma figura por ponto...')
    count_before = len(generated)
    for p in sorted(_sc['Point name'].unique()):
        sub = _sc[_sc['Point name'] == p]
        if sub.empty: continue
        fig, ax = plt.subplots(figsize=(10, 5))
        for sp in sub['Common name'].unique():
            s = sub[sub['Common name'] == sp]
            ax.scatter(s['time_float'], s['Confidence'],
                       color=color_map.get(str(sp), 'gray'), marker='o', s=40, alpha=0.8)
        ax.set_xticks(np.arange(0, 25, 1))
        ax.grid(True, linestyle='--', color='gray', alpha=0.4); ax.set_axisbelow(True)
        ax.set_title(str(p), fontsize=11, fontweight='bold')
        ax.set_xlabel('Horário'); ax.set_ylabel('Confiança')
        ax.set_xlim(0, 24); ax.set_ylim(MIN_CONFIDENCE, 1.0)
        fig.legend(leg_handles, target_species, loc='upper center', ncol=4,
                   bbox_to_anchor=(0.5, 1.02), fontsize=9, frameon=False)
        plt.tight_layout(rect=[0, 0, 1, 0.9])
        safe_p = str(p).replace(' ', '_').replace('/', '-')
        out = f'{DRIVE_PLOTS_DIR}/scatter_confidence_{safe_p}.png'
        plt.savefig(out, dpi=300); plt.show(); plt.close(); generated.append(out)
    print(f'  Salvos {len(generated) - count_before} arquivo(s).')

# ── Heatmap: species × hour ───────────────────────────────────────────────────
if PLOT_HEATMAP_SPECIES_HOUR:
    print('Gerando: mapa de calor — espécie × hora do dia...')
    _h = df.copy()
    _h['Hour'] = _h['Time'].apply(lambda t: t.hour if pd.notnull(t) else None)
    cnt = _h.groupby(['Common name', 'Hour']).size().reset_index(name='Count')
    hm  = (cnt.pivot(index='Common name', columns='Hour', values='Count')
              .reindex(index=target_species, columns=range(24)).fillna(0))
    plt.figure(figsize=(16, max(4, len(hm) * 0.35)))
    sns.heatmap(hm, annot=True, fmt='.0f', cmap='YlGnBu',
                linewidths=0.4, linecolor='black', cbar=False,
                norm=PowerNorm(gamma=0.3, vmin=0, vmax=max(hm.values.max(), 1)))
    plt.title(f'Detecções por espécie por hora  (Confiança ≥ {MIN_CONFIDENCE})', fontsize=13, fontweight='bold')
    plt.xlabel('Hora do dia'); plt.ylabel('Espécie'); plt.xticks(rotation=0)
    plt.tight_layout()
    out = f'{DRIVE_PLOTS_DIR}/heatmap_species_vs_hour.png'
    plt.savefig(out, dpi=300); plt.show(); plt.close(); generated.append(out)
    print('  Salvo: heatmap_species_vs_hour.png')

# ── Bar chart: species richness ───────────────────────────────────────────────
if PLOT_RICHNESS:
    print('Gerando: gráfico de barras de riqueza de espécies...')
    pts   = sorted(df['Point name'].unique())
    total = df.groupby('Point name')['Common name'].nunique().reindex(pts)
    unique_counts = [
        len(set(df[df['Point name'] == p]['Common name']) -
            set(df[df['Point name'] != p]['Common name']))
        for p in pts
    ]
    pdfm = pd.DataFrame({'Point name': pts, 'Total': total.values, 'Únicas': unique_counts})
    pdfm = pdfm.melt(id_vars='Point name', value_vars=['Únicas', 'Total'],
                     var_name='Categoria', value_name='Contagem')
    plt.figure(figsize=(max(8, len(pts) * 1.2), 6))
    sns.barplot(data=pdfm, x='Point name', y='Contagem', hue='Categoria',
                palette=['#66c2a5', '#fc8d62'])
    plt.title(f'Espécies únicas vs total por ponto  (Confiança ≥ {MIN_CONFIDENCE})', fontsize=13, fontweight='bold')
    plt.xlabel('Ponto'); plt.ylabel('Número de espécies')
    plt.xticks(rotation=45, ha='right'); plt.ylim(0, MAX_SPECIES_AXIS); plt.legend(title=None)
    ax = plt.gca()
    for bar in ax.patches:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.2, f'{int(h)}',
                    ha='center', va='bottom', fontweight='bold')
    plt.tight_layout()
    out = f'{DRIVE_PLOTS_DIR}/barplot_species_richness.png'
    plt.savefig(out, dpi=300); plt.show(); plt.close(); generated.append(out)
    print('  Salvo: barplot_species_richness.png')

print()
print(f'Concluído! {len(generated)} gráfico(s) salvo(s) em: {DRIVE_PLOTS_DIR}')

---
## Passo 5 — Extrair Segmentos de Áudio <a name="step-5"></a>

Este passo recorta um clipe de áudio curto para cada detecção nos seus resultados filtrados e salva no seu Google Drive.

**Como funciona:**
- Cada detecção armazena o ponto de gravação, data, hora local e a posição exata (em segundos) dentro do arquivo de áudio.
- O notebook converte os timestamps locais de volta para UTC e, em seguida, baixa a gravação do Arbimon ou a localiza na sua pasta do Drive.
- O segmento relevante é recortado e salvo como um arquivo `.wav`, organizado em uma subpasta por ponto.

**Antes de executar:**
1. Conclua os Passos 1–3 para que os resultados estejam carregados e filtrados.
2. Preencha as células de configuração abaixo e execute-as de cima para baixo.

> **Dica:** Use o filtro de espécies no Passo 3 para restringir quais detecções extrair antes de executar este passo.

In [ ]:
#@title ⚙️ Configurações de extração { display-mode: "form" }

#@markdown **Origem do áudio** — onde estão as gravações originais?
AUDIO_SOURCE = "arbimon"  #@param ["arbimon", "google_drive"]

#@markdown ---
#@markdown ### Se a origem do áudio = Google Drive
#@markdown Caminho completo para a pasta que contém os arquivos de áudio no Drive.
#@markdown Subpastas com os nomes dos pontos são pesquisadas automaticamente.
DRIVE_AUDIO_DIR = "/content/drive/MyDrive/audio"  #@param {type:"string"}

#@markdown ---
#@markdown **Pasta de saída** — onde os clipes extraídos serão salvos no Drive.
DRIVE_SEGMENTS_DIR = "/content/drive/MyDrive/segments"  #@param {type:"string"}

#@markdown ---
#@markdown **Espécies a extrair** — deixe em branco para extrair todas as espécies nos resultados filtrados.
#@markdown Separe os nomes com vírgulas. Exemplo: `Robin, Blackbird`
EXTRACT_SPECIES = ""  #@param {type:"string"}

#@markdown ---
#@markdown **Margem (segundos)** — áudio extra adicionado antes e depois de cada janela de detecção.
SEGMENT_PADDING_S = 0.0  #@param {type:"number"}

#@markdown **Taxa de amostragem de saída (Hz)** — taxa para os arquivos WAV salvos.
#@markdown Use a mesma taxa do seu modelo (ex.: 48000 para BirdNET, 32000 para Google Perch).
EXTRACT_SR = 48000  #@param {type:"integer"}

import os
os.makedirs(DRIVE_SEGMENTS_DIR, exist_ok=True)
print(f'Origem do áudio    : {AUDIO_SOURCE}')
print(f'Pasta de saída     : {DRIVE_SEGMENTS_DIR}')
print(f'Margem             : {SEGMENT_PADDING_S} s')
print(f'Taxa de amostragem : {EXTRACT_SR} Hz')
if EXTRACT_SPECIES.strip():
    print(f'Filtro de espécies : {[x.strip() for x in EXTRACT_SPECIES.split(",") if x.strip()]}')
else:
    print('Filtro de espécies : todas (dos resultados filtrados no Passo 3)')

In [ ]:
#@title 🔑 Fazer login no Arbimon (ignore se usar o Google Drive) { display-mode: "form" }
#@markdown Execute esta célula apenas se a origem do áudio for **Arbimon**.
#@markdown Se você executou o notebook Analisador nesta sessão, já está logado — pode ignorar esta célula.
CREDENTIALS_PATH = '/content/drive/MyDrive/rfcx/.rfcx_credentials'  #@param {type:"string"}

if AUDIO_SOURCE == 'arbimon':
    import rfcx
    client_extract = rfcx.Client()
    if os.path.exists(CREDENTIALS_PATH):
        print('Arquivo de credenciais encontrado — fazendo login automaticamente...')
        client_extract.authenticate(persisted_credentials_path=CREDENTIALS_PATH)
    else:
        print('Nenhuma credencial salva encontrada.')
        print('A URL will appear below — open it in your browser and log in with your Arbimon account.')
        print('-' * 60)
        client_extract.authenticate(persisted_credentials_path=CREDENTIALS_PATH)
    print('\nLogin no Arbimon realizado com sucesso.')
else:
    print('Origem do áudio é o Google Drive — login no Arbimon não necessário. Você pode ignorar esta célula.')

In [ ]:
#@title ⬇️ Localizar ou baixar gravações { display-mode: "form" }
#@markdown Identifica quais gravações originais contêm as detecções selecionadas,
#@markdown depois as baixa do Arbimon ou as localiza na sua pasta do Drive.

import re, glob, datetime
import pandas as pd

# Build extraction subset
df_ext = df.copy()
if EXTRACT_SPECIES.strip():
    _ext_sp = [x.strip() for x in EXTRACT_SPECIES.split(',') if x.strip()]
    df_ext  = df_ext[df_ext['Common name'].isin(_ext_sp)]

if df_ext.empty:
    raise ValueError(
        'No detections match the extraction criteria.\n'
        'Check your species list and the filters applied in Step 3.'
    )

print(f'Detecções para extrair   : {len(df_ext)}')

def _utc_hours(utc_str):
    try:
        return int(str(utc_str).replace('UTC', '').strip() or 0)
    except ValueError:
        return 0

def _rec_datetime(row):
    # Returns the datetime of the recording start that contains this detection.
    # For Arbimon: converts local time to UTC using utc_offset.
    # For Google Drive: keeps local time (filenames use local time).
    try:
        local_dt = datetime.datetime.combine(row['Date'], row['Time'])
        if AUDIO_SOURCE == 'arbimon':
            local_dt = local_dt - datetime.timedelta(hours=_utc_hours(row.get('utc_offset', 'UTC+0')))
        return local_dt
    except Exception:
        return None

df_ext = df_ext.copy()
df_ext['_rec_dt'] = df_ext.apply(_rec_datetime, axis=1)

if AUDIO_SOURCE == 'arbimon':
    unique_recs = df_ext[['stream_id', '_rec_dt']].dropna().drop_duplicates()
else:
    unique_recs = df_ext[['_rec_dt']].dropna().drop_duplicates()

print(f'Gravações únicas necessárias : {len(unique_recs)}')

AUDIO_TEMP_DIR = '/content/audio_extract'
os.makedirs(AUDIO_TEMP_DIR, exist_ok=True)

audio_file_map = {}  # key → local audio file path

# ── Arbimon: one download call per needed recording ──────────────────────────
if AUDIO_SOURCE == 'arbimon':
    dest = AUDIO_TEMP_DIR
    os.makedirs(dest, exist_ok=True)

    def _arbimon_file_dt(filepath):
        name = os.path.basename(filepath)
        m = re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', name)
        if m:
            return datetime.datetime(*[int(g) for g in m.groups()])
        return None

    for _, rec in unique_recs.iterrows():
        sid = str(rec['stream_id'])
        t0  = rec['_rec_dt']
        key = (sid, t0)

        # Check if already downloaded in a previous iteration
        existing = (glob.glob(f'{dest}/**/*.wav',  recursive=True) +
                    glob.glob(f'{dest}/**/*.flac', recursive=True))
        already = next(
            (f for f in existing
             if sid in os.path.basename(f)
             and (fdt := _arbimon_file_dt(f)) is not None
             and abs((fdt - t0).total_seconds()) < 2),
            None
        )
        if already:
            audio_file_map[key] = already
            print(f'  Em cache : {os.path.basename(already)}')
            continue

        # Small window: 5 s before to 55 s after the recording start.
        # Stays within a single 60-second recording and avoids the API's
        # exclusive-lower-bound behaviour at exactly t0.
        q_min = t0 - datetime.timedelta(seconds=5)
        q_max = t0 + datetime.timedelta(seconds=55)
        try:
            client_extract.download_segments(
                dest_path=dest, stream=sid,
                min_date=q_min, max_date=q_max, parallel=False,
            )
        except Exception as e:
            print(f'  ERROR — stream {sid} at {t0}: {e}')
            continue

        all_downloaded = (glob.glob(f'{dest}/**/*.wav',  recursive=True) +
                          glob.glob(f'{dest}/**/*.flac', recursive=True))
        matched = next(
            (f for f in all_downloaded
             if sid in os.path.basename(f)
             and (fdt := _arbimon_file_dt(f)) is not None
             and abs((fdt - t0).total_seconds()) < 2),
            None
        )
        if matched:
            audio_file_map[key] = matched
            print(f'  Baixado : {os.path.basename(matched)}')
        else:
            print(f'  AVISO: nenhum arquivo encontrado para stream {sid} em {t0}')

# ── Google Drive: match files by datetime in filename ────────────────────────
elif AUDIO_SOURCE == 'google_drive':
    all_audio = (
        glob.glob(f'{DRIVE_AUDIO_DIR}/**/*.wav',  recursive=True) +
        glob.glob(f'{DRIVE_AUDIO_DIR}/**/*.flac', recursive=True) +
        glob.glob(f'{DRIVE_AUDIO_DIR}/**/*.mp3',  recursive=True)
    )
    print(f'Arquivos de áudio no Drive : {len(all_audio)}')

    def _file_dt(filepath):
        name = os.path.basename(filepath)
        m = re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', name)
        if m:
            return datetime.datetime(*[int(g) for g in m.groups()])
        m = re.search(r'(\d{8})_(\d{6})', name)
        if m:
            d, t = m.group(1), m.group(2)
            return datetime.datetime(int(d[:4]), int(d[4:6]), int(d[6:8]),
                                     int(t[:2]), int(t[2:4]), int(t[4:6]))
        return None

    for _, rec in unique_recs.iterrows():
        t0  = rec['_rec_dt']
        key = t0
        matched = next(
            (f for f in all_audio
             if (fdt := _file_dt(f)) and abs((fdt - t0).total_seconds()) < 5),
            None
        )
        if matched:
            audio_file_map[key] = matched
            print(f'  Correspondido : {os.path.basename(matched)}')
        else:
            print(f'  AVISO: nenhum arquivo de áudio encontrado para gravação em {t0}')

print(f'\nArquivos de áudio prontos : {len(audio_file_map)} / {len(unique_recs)}')

In [ ]:
#@title ✂️ Extrair e salvar segmentos de áudio { display-mode: "form" }
#@markdown Recorta cada detecção da sua gravação e salva como arquivo WAV.
#@markdown
#@markdown Os nomes dos arquivos seguem o padrão:
#@markdown `PONTO_YYYYMMDD_HHMMSS_INICIOs_CONF_ROTULO.wav`

import librosa, soundfile as sf
import numpy as np

saved   = 0
skipped = 0
errors  = 0

for _, row in df_ext.iterrows():
    if AUDIO_SOURCE == 'arbimon':
        key = (str(row['stream_id']), row['_rec_dt'])
    else:
        key = row['_rec_dt']
    audio_path = audio_file_map.get(key)

    if not audio_path:
        errors += 1
        continue

    start_s    = max(0.0, float(row['start_time']) - SEGMENT_PADDING_S)
    end_s      = float(row['end_time']) + SEGMENT_PADDING_S
    conf       = float(row['Confidence'])

    site_safe  = str(row['Point name']).replace(' ', '_')
    label_safe = str(row['Common name']).replace(' ', '_').replace('/', '-')
    date_str   = str(row['Date']).replace('-', '')
    time_str   = str(row['Time']).replace(':', '')[:6]
    conf_str   = f'{conf:.2f}'
    out_name   = f'{site_safe}_{date_str}_{time_str}_{int(start_s)}_{int(end_s)}s_{conf_str}_{label_safe}.wav'

    site_out = os.path.join(DRIVE_SEGMENTS_DIR, site_safe)
    os.makedirs(site_out, exist_ok=True)
    out_path = os.path.join(site_out, out_name)

    if os.path.exists(out_path):
        skipped += 1
        continue

    try:
        audio, _ = librosa.load(audio_path, sr=EXTRACT_SR, mono=True,
                                offset=start_s, duration=end_s - start_s)
    except Exception as e:
        print(f'  ERROR loading {os.path.basename(audio_path)}: {e}')
        errors += 1
        continue

    sf.write(out_path, audio, EXTRACT_SR)
    saved += 1

print(f'Segmentos salvos        : {saved}')
if skipped:
    print(f'Já existiam             : {skipped} (ignorados)')
if errors:
    print(f'Ignorados (sem áudio)   : {errors}')
print(f'Pasta de saída          : {DRIVE_SEGMENTS_DIR}')

---
## Passo 6 — Revisar Segmentos Extraídos <a name="step-6"></a>

Navegue pelos clipes de áudio extraídos no Passo 5, um de cada vez.
Cada segmento é exibido como um espectrograma com seu rótulo e pontuação de confiança.
Você pode ouvi-lo com o player de áudio integrado e marcá-lo como **verdadeiro** (detecção genuína) ou **falso** (falso positivo).

- Marcar um segmento move o arquivo para a subpasta `true/` ou `false/` dentro da pasta de segmentos.
- Use **← Anterior** e **Próximo →** para navegar sem tomar uma decisão.
- Arquivos já revisados (dentro de `true/` ou `false/`) são excluídos da lista automaticamente a cada execução da célula.
- Os **controles do espectrograma** (tipo, faixa de frequência, piso em dB) atualizam a visualização instantaneamente sem recarregar o áudio.

> Execute o Passo 5 primeiro para que a pasta de segmentos seja preenchida, e então execute as duas células abaixo.

In [6]:
#@title ⚙️ Configurações de revisão { display-mode: "form" }
#@markdown Caminho completo para a pasta com os segmentos extraídos.
#@markdown Use o mesmo valor de **DRIVE_SEGMENTS_DIR** do Passo 5.
REVIEW_DIR = "/content/drive/MyDrive/segments"  #@param {type:"string"}

#@markdown ---
#@markdown ### Padrões do espectrograma
#@markdown Estes definem os valores iniciais exibidos no revisor.
#@markdown Você pode ajustar todos eles em tempo real dentro da GUI sem re-executar esta célula.

#@markdown **Tipo de espectrograma** — `mel` usa escala de frequência perceptual; `fft` usa escala linear em Hz.
SPEC_TYPE = "mel"  #@param ["mel", "fft"]

#@markdown **Frequência mínima (Hz)** — limite inferior do eixo de frequência.
FREQ_MIN_HZ = 0  #@param {type:"integer"}

#@markdown **Frequência máxima (Hz)** — limite superior. Defina **0** para usar a frequência de Nyquist (metade da taxa de amostragem).
FREQ_MAX_HZ = 0  #@param {type:"integer"}

#@markdown **Piso em dB** — menor valor exibido na escala de cores (ex.: −80). Intervalo típico: −100 a −40.
DB_MIN = -80  #@param {type:"integer"}

import os
os.makedirs(os.path.join(REVIEW_DIR, 'true'),  exist_ok=True)
os.makedirs(os.path.join(REVIEW_DIR, 'false'), exist_ok=True)

import glob as _g
_all  = _g.glob(f'{REVIEW_DIR}/**/*.wav', recursive=True)
_skip = {os.path.join(REVIEW_DIR, 'true'), os.path.join(REVIEW_DIR, 'false')}
_pending = [f for f in _all
            if not any(os.path.commonpath([f, s]) == s for s in _skip)]

print(f'Pasta de segmentos   : {REVIEW_DIR}')
print(f'Aguardando revisão   : {len(_pending)} segmento(s)')
print(f'Já verdadeiros   : {len(_g.glob(os.path.join(REVIEW_DIR, "true",  "*.wav")))}')
print(f'Já falsos        : {len(_g.glob(os.path.join(REVIEW_DIR, "false", "*.wav")))}')
print(f'Tipo de espectrograma : {SPEC_TYPE}')
print(f'Faixa de frequência   : {FREQ_MIN_HZ} – {"Nyquist" if FREQ_MAX_HZ == 0 else str(FREQ_MAX_HZ) + " Hz"}')
print(f'Piso em dB            : {DB_MIN} dB')

Segments folder  : /content/drive/MyDrive/segments
Pending review   : 2 segment(s)
Already true     : 19
Already false    : 0
Spectrogram type : mel
Freq range       : 0 – Nyquist
dB floor         : -80 dB


In [ ]:
#@title 🎧 Revisor de segmentos { display-mode: "form" }
#@markdown Execute esta célula para abrir o revisor interativo.
#@markdown Use **Verdadeiro / Falso** para classificar um segmento e avançar.
#@markdown Use **← Anterior / Próximo →** para navegar sem classificar.
#@markdown Ajuste os controles do espectrograma para alterar a visualização — o player de áudio não é afetado.
#@markdown Quando **Falso** for pressionado, insira o rótulo correto antes de confirmar.

import os, glob, shutil, re, io, base64
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# ── Collect unreviewed segments ───────────────────────────────────────────────
def _collect(base):
    skip = {os.path.join(base, 'true'), os.path.join(base, 'false')}
    return sorted(
        f for f in glob.glob(f'{base}/**/*.wav', recursive=True)
        if not any(os.path.commonpath([f, s]) == s for s in skip)
    )

# ── Parse label + confidence from filename ────────────────────────────────────
def _parse(filepath):
    stem = os.path.splitext(os.path.basename(filepath))[0]
    m = re.search(r'_(\d\.\d+)_(.+)$', stem)
    if m:
        return m.group(2).replace('_', ' '), float(m.group(1))
    return stem, None

# ── State ─────────────────────────────────────────────────────────────────────
_state = {'idx': 0, 'segs': _collect(REVIEW_DIR)}

# ── Spectrogram controls ──────────────────────────────────────────────────────
_W_ctrl = {'style': {'description_width': 'initial'}}

_dd_type = widgets.Dropdown(
    options=[('Mel', 'mel'), ('FFT (linear)', 'fft')],
    value=globals().get('SPEC_TYPE', 'mel'),
    description='Type:',
    layout=widgets.Layout(width='165px'),
    **_W_ctrl,
)
_int_fmin = widgets.BoundedIntText(
    value=globals().get('FREQ_MIN_HZ', 0), min=0, max=96000, step=100,
    description='Min Hz:',
    layout=widgets.Layout(width='155px'),
    **_W_ctrl,
)
_int_fmax = widgets.BoundedIntText(
    value=globals().get('FREQ_MAX_HZ', 0), min=0, max=96000, step=100,
    description='Max Hz:',
    layout=widgets.Layout(width='155px'),
    **_W_ctrl,
)
_int_db = widgets.BoundedIntText(
    value=globals().get('DB_MIN', -80), min=-120, max=-20, step=5,
    description='dB floor:',
    layout=widgets.Layout(width='160px'),
    **_W_ctrl,
)
_ctrl_row = widgets.HBox(
    [_dd_type, _int_fmin, _int_fmax, _int_db,
     widgets.Label('(Hz máx = 0 → Nyquist)', layout=widgets.Layout(margin='6px 0 0 4px'))],
    layout=widgets.Layout(justify_content='center', margin='4px 0', flex_wrap='wrap'),
)

# ── Display widgets ───────────────────────────────────────────────────────────
_spec_widget  = widgets.Image(value=b'', format='png',
                               layout=widgets.Layout(width='720px', height='auto'))
_audio_widget = widgets.HTML(value='')
_lbl_prog     = widgets.HTML(value='')
_lbl_error    = widgets.HTML(value='')

# ── Navigation row ────────────────────────────────────────────────────────────
_BW = widgets.Layout(width='120px', height='36px')
_btn_prev  = widgets.Button(description='← Anterior',   layout=widgets.Layout(width='100px', height='36px'))
_btn_true  = widgets.Button(description='✔  Verdadeiro',  button_style='success', layout=_BW)
_btn_false = widgets.Button(description='✘  Falso', button_style='danger',  layout=_BW)
_btn_next  = widgets.Button(description='Próximo →',   layout=widgets.Layout(width='100px', height='36px'))

_nav_row = widgets.HBox(
    [_btn_prev, _btn_true, _btn_false, _btn_next],
    layout=widgets.Layout(justify_content='center', margin='8px 0'),
)

# ── False-label input row (hidden until False is pressed) ─────────────────────
_input_label       = widgets.Text(
    placeholder='Rótulo correto  (deixe em branco → "desconhecido")',
    layout=widgets.Layout(width='340px', height='36px'),
)
_btn_confirm_false = widgets.Button(description='Confirmar', button_style='warning',
                                     layout=widgets.Layout(width='100px', height='36px'))
_btn_cancel        = widgets.Button(description='Cancelar',
                                     layout=widgets.Layout(width='80px',  height='36px'))

_label_row = widgets.VBox([
    widgets.HTML('<span style="font-size:12px">Insira o rótulo correto para este segmento:</span>'),
    widgets.HBox(
        [_input_label, _btn_confirm_false, _btn_cancel],
        layout=widgets.Layout(justify_content='center', margin='4px 0'),
    ),
], layout=widgets.Layout(align_items='center', margin='4px 0', display='none'))

# ── Full UI ───────────────────────────────────────────────────────────────────
_ui = widgets.VBox([
    _lbl_prog,
    _ctrl_row,
    _spec_widget,
    _audio_widget,
    _lbl_error,
    _label_row,
    _nav_row,
], layout=widgets.Layout(align_items='center'))

# ── Spectrogram rendering ─────────────────────────────────────────────────────
def _show_spec():
    segs = _state['segs']
    if not segs:
        _spec_widget.value = b''
        return
    filepath = segs[_state['idx']]
    label, conf = _parse(filepath)
    conf_str  = f'{conf:.3f}' if conf is not None else '?'
    spec_type = _dd_type.value
    fmin_hz   = _int_fmin.value
    db_floor  = _int_db.value
    _lbl_error.value = ''
    try:
        y, sr   = librosa.load(filepath, sr=None, mono=True)
        fmax_hz = _int_fmax.value if _int_fmax.value > 0 else sr // 2
        fig, ax = plt.subplots(figsize=(9, 4))
        if spec_type == 'mel':
            S   = librosa.feature.melspectrogram(
                y=y, sr=sr, n_mels=128, fmin=fmin_hz, fmax=fmax_hz)
            Sd  = librosa.power_to_db(S, ref=np.max)
            img = librosa.display.specshow(
                Sd, sr=sr, x_axis='time', y_axis='mel', ax=ax,
                fmin=fmin_hz, fmax=fmax_hz, vmin=db_floor, vmax=0)
        else:
            D   = librosa.stft(y)
            Sd  = librosa.amplitude_to_db(np.abs(D), ref=np.max)
            img = librosa.display.specshow(
                Sd, sr=sr, x_axis='time', y_axis='hz', ax=ax,
                vmin=db_floor, vmax=0)
            ax.set_ylim(fmin_hz, fmax_hz)
        ax.set_title(f'{label}   |   score: {conf_str}', fontsize=13, fontweight='bold')
        fig.colorbar(img, ax=ax, format='%+2.0f dB')
        plt.tight_layout()
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        _spec_widget.value = buf.read()
    except Exception as e:
        plt.close('all')
        _spec_widget.value = b''
        _lbl_error.value = f'<span style="color:red">Erro no espectrograma: {e}</span>'

def _show_audio():
    segs = _state['segs']
    if not segs:
        _audio_widget.value = ''
        return
    try:
        with open(segs[_state['idx']], 'rb') as f:
            b64 = base64.b64encode(f.read()).decode()
        _audio_widget.value = (
            f'<audio controls style="width:720px;margin-top:4px">'
            f'<source src="data:audio/wav;base64,{b64}" type="audio/wav">'
            f'</audio>'
        )
    except Exception as e:
        _audio_widget.value = f'<span style="color:red">Erro no áudio: {e}</span>'

def _show():
    segs = _state['segs']
    n    = len(segs)
    if n == 0:
        _lbl_prog.value     = '<b>Todos os segmentos foram revisados.</b>'
        _spec_widget.value  = b''
        _audio_widget.value = ''
        return
    _lbl_prog.value = (
        f'<span style="font-size:13px">Segmento <b>{_state["idx"] + 1}</b> de <b>{n}</b></span>'
    )
    _show_spec()
    _show_audio()

# ── File operations ───────────────────────────────────────────────────────────
def _move(verdict, subfolder=None):
    segs = _state['segs']
    if not segs:
        return
    dest_dir = os.path.join(REVIEW_DIR, verdict, subfolder) if subfolder \
               else os.path.join(REVIEW_DIR, verdict)
    os.makedirs(dest_dir, exist_ok=True)
    src = segs.pop(_state['idx'])
    shutil.move(src, os.path.join(dest_dir, os.path.basename(src)))
    if _state['idx'] >= len(segs) and _state['idx'] > 0:
        _state['idx'] -= 1
    _show()

def _nav(delta):
    n = len(_state['segs'])
    if n == 0:
        return
    _state['idx'] = max(0, min(n - 1, _state['idx'] + delta))
    _show()

# ── Button handlers ───────────────────────────────────────────────────────────
def _show_label_input(_):
    _input_label.value        = ''
    _label_row.layout.display = ''
    _nav_row.layout.display   = 'none'

def _on_confirm_false(_):
    raw      = _input_label.value.strip()
    subfolder = raw.replace(' ', '_').replace('/', '-') if raw else 'desconhecido'
    _label_row.layout.display = 'none'
    _nav_row.layout.display   = ''
    _move('false', subfolder)

def _on_cancel(_):
    _label_row.layout.display = 'none'
    _nav_row.layout.display   = ''

_btn_true.on_click(lambda _: _move('true'))
_btn_false.on_click(_show_label_input)
_btn_confirm_false.on_click(_on_confirm_false)
_btn_cancel.on_click(_on_cancel)
_btn_prev.on_click(lambda _: _nav(-1))
_btn_next.on_click(lambda _: _nav(+1))

# ── Spec-change observers ─────────────────────────────────────────────────────
def _on_spec_change(change):
    if _state['segs']:
        _show_spec()

for _ctrl in (_dd_type, _int_fmin, _int_fmax, _int_db):
    _ctrl.observe(_on_spec_change, names='value')

# ── Launch ────────────────────────────────────────────────────────────────────
_show()
display(_ui)